# 03 — Agent Demo: Base vs Classifier-Adapted

The core question this notebook answers:

> **What does a fine-tuned intent classifier actually buy you in a production agentic loop?**

We run 5 realistic IT support queries through two agents side-by-side and measure the difference in tool calls, LLM calls, and latency.

## 1. Architecture overview

```
        BASE AGENT                              ADAPTED AGENT
        ──────────────────────────────          ──────────────────────────────────────

   [Employee request]                      [Employee request]
            │                                        │
            │                             ┌──────────▼──────────────────────┐
            │                             │  Layer 1: SFT Classifier        │
            │                             │  open-mistral-nemo              │
            │                             │  fine-tuned on Veridian tickets │
            │                             └──────────┬──────────────────────┘
            │                                        │
            │                              intent label + confidence score
            │                             (e.g. "access_request", 95%)
            │                                        │
            │                             ┌──────────▼──────────────────────┐
            │                             │  Tool Router                    │
            │                             │  "adapted" → 2 tools only       │
            │                             │  injects intent into sys prompt │
            │                             └──────────┬──────────────────────┘
            │                                        │
   ┌────────▼────────────────────┐       ┌──────────▼──────────────────────┐
   │  mistral-large-latest       │       │  mistral-large-latest           │
   │                             │       │                                 │
   │  tools available: 3         │       │  tools available: 2             │
   │  • search_knowledge_base    │       │  • create_ticket                │
   │  • create_ticket            │       │  • get_escalation_policy        │
   │  • get_escalation_policy    │       │                                 │
   │                             │       │  model skips KB search —        │
   │  model figures out intent   │       │  classifier already routed it   │
   │  by trial and error         │       │                                 │
   └────────┬────────────────────┘       └──────────┬──────────────────────┘
            │                                        │
     [Helpdesk reply]                        [Helpdesk reply]
```

**What we measure across the 5 demo scenarios:**
- Tool calls and LLM calls (fewer = better)
- Total latency in ms (classifier adds ~50–150 ms, loop subtracts more)
- Whether each agent's intent matches the ground truth label for the scenario

In [1]:
!uv add mistralai python-dotenv rich ipywidgets pandas

Resolved 151 packages in 0.80ms
Audited 145 packages in 1ms


In [2]:
## 1. Architecture overview — rendered in the next markdown cell

# ── Imports ────────────────────────────────────────────────────────────────
import json
import os
import sys
from pathlib import Path

# Make agents/ importable when running from mistral-it-agent/
_HERE = Path(".").resolve()
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))

import ipywidgets as widgets
import pandas as pd
from IPython.display import clear_output, display
from mistralai.client import Mistral
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

from agents.base_agent import BaseAgent
from agents.adapted_agent import AdaptedAgent

console = Console(width=120)
console.print("[bold green]Imports OK.[/bold green]")

Imports OK.

In [3]:
## 2. Setup — API keys, classifier model ID, agent initialisation

# ── API keys ───────────────────────────────────────────────────────────────
try:
    from google.colab import userdata  # type: ignore
    MISTRAL_API_KEY  = userdata.get("MISTRAL_API_KEY")
    TOGETHER_API_KEY = userdata.get("TOGETHER_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv(override=True)
    MISTRAL_API_KEY  = os.getenv("MISTRAL_API_KEY")
    TOGETHER_API_KEY = os.getenv("TOGETHER_API_KEY")

if not MISTRAL_API_KEY:
    raise EnvironmentError(
        "MISTRAL_API_KEY not set.\n"
        "Local: add it to a .env file in mistral-it-agent/.\n"
        "Colab: add it via Secrets (key icon in the left sidebar)."
    )

# ── Classifier model ID (written by 02_train_classifier.ipynb) ────────────
MODEL_ID_FILE = Path("data/classifier_model_id.txt")
if MODEL_ID_FILE.exists():
    CLASSIFIER_MODEL_ID = MODEL_ID_FILE.read_text().strip()
    console.print(f"[bold]Classifier model:[/bold] [cyan]{CLASSIFIER_MODEL_ID}[/cyan]")
else:
    CLASSIFIER_MODEL_ID = None
    console.print("[yellow]No trained classifier found — using keyword mock.[/yellow]")
    console.print("[dim]Run 02_train_classifier.ipynb first to use the real model.[/dim]")

# ── Initialise clients ─────────────────────────────────────────────────────
from mistralai.client import Mistral
mistral_client = Mistral(api_key=MISTRAL_API_KEY)

classifier_client = None
if CLASSIFIER_MODEL_ID and TOGETHER_API_KEY:
    from together import Together
    classifier_client = Together(api_key=TOGETHER_API_KEY)
    console.print("[bold green]Together.ai classifier client ready.[/bold green]")
elif CLASSIFIER_MODEL_ID and not TOGETHER_API_KEY:
    console.print("[yellow]TOGETHER_API_KEY not set — classifier will use keyword mock.[/yellow]")
    CLASSIFIER_MODEL_ID = None

base_agent = BaseAgent(client=mistral_client, model="mistral-large-latest")
adapted_agent = AdaptedAgent(
    client=mistral_client,
    classifier_model_id=CLASSIFIER_MODEL_ID,
    model="mistral-large-latest",
    classifier_client=classifier_client,
)
console.print("[bold green]Both agents ready.[/bold green]")

Classifier model: crowe3555_2a6c/Mistral-7B-v0.1-27e98401-cdd14f64

Together.ai classifier client ready.

Both agents ready.

In [4]:
# ── Ensure the classifier endpoint is running ─────────────────────────────
# Fine-tuned LoRA models require a dedicated Together.ai endpoint.
# This cell starts it if stopped and waits until ready.
# endpoint_id  — used for retrieve/update API calls
# endpoint_name — used as model= in chat.completions.create()

import time as _time

_ENDPOINT_ID_FILE   = Path("data/endpoint_id.txt")
_ENDPOINT_NAME_FILE = Path("data/endpoint_name.txt")

if classifier_client and _ENDPOINT_ID_FILE.exists() and _ENDPOINT_NAME_FILE.exists():
    _endpoint_id   = _ENDPOINT_ID_FILE.read_text().strip()
    _endpoint_name = _ENDPOINT_NAME_FILE.read_text().strip()

    if _endpoint_id:
        _ep = classifier_client.endpoints.retrieve(_endpoint_id)
        console.print(f"Endpoint [cyan]{_endpoint_name}[/cyan] — state: [yellow]{_ep.state}[/yellow]")

        if _ep.state == "STOPPED":
            console.print("Starting endpoint…")
            classifier_client.endpoints.update(_endpoint_id, state="STARTED")

        if _ep.state != "STARTED":
            while True:
                _ep = classifier_client.endpoints.retrieve(_endpoint_id)
                console.print(f"  state = [yellow]{_ep.state}[/yellow]")
                if _ep.state == "STARTED":
                    break
                if _ep.state == "ERROR":
                    raise RuntimeError(f"Endpoint entered error state.")
                _time.sleep(20)

        console.print("[bold green]Endpoint ready.[/bold green]")
else:
    console.print("[dim]No endpoint files found — using keyword mock classifier.[/dim]")

Endpoint crowe3555_2a6c/Mistral-7B-v0.1-27e98401-cdd14f64 — state: STARTED

Endpoint ready.

## 3. Test set

The held-out test set from `01_data_prep.ipynb` (15 % of all examples,
stratified by intent) is loaded here. Running both agents across the
full test set gives a statistically meaningful accuracy comparison rather
than relying on a handful of hand-picked scenarios.


In [5]:
DATA_DIR = Path("data")

test_records = []
with open(DATA_DIR / "test.jsonl") as _f:
    for _line in _f:
        _ex   = json.loads(_line)
        _msgs = _ex["messages"]
        test_records.append({
            "text":         _msgs[1]["content"],
            "ground_truth": _msgs[2]["content"],
        })

console.print(f"Loaded [bold]{len(test_records)}[/bold] test examples "
              f"across [bold]{len({r['ground_truth'] for r in test_records})}[/bold] intent classes.")


Loaded 64 test examples across 8 intent classes.

## 4. Comparison functions

`_evaluate_pair(text, ground_truth)` runs both agents on a single example
and returns a metrics dict without printing anything — used by the bulk
test set runner in Section 5.

`run_comparison(scenario)` runs both agents and renders a full side-by-side
panel — used by the interactive cell in Section 6 for ad-hoc queries.


In [6]:
def _evaluate_pair(text: str, ground_truth: str) -> dict:
    """
    Run base_agent and adapted_agent on a single example.
    Returns a metrics dict without printing anything.
    """
    base_result    = base_agent.run(text)
    adapted_result = adapted_agent.run(text)

    base_intent = _infer_base_intent(base_result)
    # Base agent handles general_question by answering directly (no tool call).
    # Treat an absent tool-based intent as general_question when that is the
    # ground truth, so the base agent is not unfairly penalised.
    if ground_truth == "general_question" and base_intent == "— (no routing tool called)":
        base_intent = "general_question"

    adapted_intent = adapted_result["classifier_intent"]
    adapted_conf   = adapted_result["classifier_confidence"]

    base_prompt     = base_result.get("prompt_tokens", 0)
    base_completion = base_result.get("completion_tokens", 0)
    adpt_prompt     = adapted_result.get("prompt_tokens", 0)
    adpt_completion = adapted_result.get("completion_tokens", 0)
    cls_prompt      = adapted_result.get("classifier_prompt_tokens", 0)
    cls_completion  = adapted_result.get("classifier_completion_tokens", 0)

    return {
        "ground_truth":          ground_truth,
        "base_intent":           base_intent,
        "adapted_intent":        adapted_intent,
        "base_correct":          base_intent == ground_truth,
        "adapted_correct":       adapted_intent == ground_truth,
        "base_tool_calls":       len(base_result["tools_called"]),
        "adapted_tool_calls":    len(adapted_result["tools_called"]),
        "base_latency_ms":       base_result["latency_ms"],
        "adapted_latency_ms":    adapted_result["latency_ms"],
        "classifier_latency_ms": adapted_result["classifier_latency_ms"],
        "base_cost_usd":         _compute_cost(base_prompt, base_completion),
        "adapted_cost_usd":      _compute_cost(adpt_prompt, adpt_completion, cls_prompt, cls_completion),
        "classifier_confidence": adapted_conf,
    }


# Pricing constants (La Plateforme / Together.ai, as of 2025)
_LARGE_INPUT_PER_M  = 2.00   # mistral-large-latest, $/M input tokens
_LARGE_OUTPUT_PER_M = 6.00   # mistral-large-latest, $/M output tokens
_CLS_PER_M          = 0.20   # Together.ai fine-tuned Mistral-7B, $/M tokens (in+out)


def _compute_cost(prompt_tokens: int, completion_tokens: int,
                  cls_prompt_tokens: int = 0, cls_completion_tokens: int = 0) -> float:
    """Return total USD cost for one agent run."""
    loop_cost = (prompt_tokens * _LARGE_INPUT_PER_M
                 + completion_tokens * _LARGE_OUTPUT_PER_M) / 1_000_000
    cls_cost  = (cls_prompt_tokens + cls_completion_tokens) * _CLS_PER_M / 1_000_000
    return loop_cost + cls_cost


def _infer_base_intent(result: dict) -> str:
    """
    Best-effort: extract the intent the base agent implicitly routed to by
    inspecting the category/intent arguments it passed to tools.
    """
    for tr in result["tool_results"]:
        if tr["name"] in ("get_escalation_policy", "create_ticket"):
            try:
                args = tr["arguments"]
                if isinstance(args, str):
                    args = json.loads(args)
                return args.get("category", "—")
            except (json.JSONDecodeError, TypeError, AttributeError):
                pass
    return "— (no routing tool called)"


def run_comparison(scenario: dict) -> dict:
    """
    Run base_agent and adapted_agent on scenario["text"].

    Prints:
      1. A side-by-side metric table (intent, tools, LLM calls, latency, cost, response preview)
      2. A "What the SFT classifier changed" summary panel

    Returns a flat dict for aggregation into the summary DataFrame.
    """
    console.rule(f"[bold]{scenario['name']}[/bold]")
    console.print(Panel(
        scenario["text"],
        title="[bold blue]Employee request[/bold blue]",
        border_style="blue",
    ))

    # ── Run both agents ────────────────────────────────────────────────────
    base_result    = base_agent.run(scenario["text"])
    adapted_result = adapted_agent.run(scenario["text"])

    # ── Extract comparison values ──────────────────────────────────────────
    ground_truth   = scenario["ground_truth"]
    base_intent    = _infer_base_intent(base_result)
    adapted_intent = adapted_result["classifier_intent"]
    adapted_conf   = adapted_result["classifier_confidence"]
    base_correct     = base_intent == ground_truth
    adapted_correct  = adapted_intent == ground_truth

    # ── Token usage & cost ─────────────────────────────────────────────────
    base_prompt      = base_result.get("prompt_tokens", 0)
    base_completion  = base_result.get("completion_tokens", 0)
    base_cost        = _compute_cost(base_prompt, base_completion)

    adpt_prompt      = adapted_result.get("prompt_tokens", 0)
    adpt_completion  = adapted_result.get("completion_tokens", 0)
    cls_prompt       = adapted_result.get("classifier_prompt_tokens", 0)
    cls_completion   = adapted_result.get("classifier_completion_tokens", 0)
    adapted_cost     = _compute_cost(adpt_prompt, adpt_completion, cls_prompt, cls_completion)

    def _preview(text: str, limit: int = 200) -> str:
        return (text[:limit] + "…") if len(text) > limit else text

    # ── Classifier model display info ──────────────────────────────────────
    if adapted_agent.classifier_model_id:
        _cls_model  = f"Fine-tuned Mistral-7B-v0.2\n[dim]{adapted_agent.classifier_model_id}[/dim]"
        _cls_params = "~7B"
    else:
        _cls_model  = "[dim]Keyword mock (no model)[/dim]"
        _cls_params = "[dim]N/A[/dim]"

    # ── Side-by-side table ─────────────────────────────────────────────────
    t = Table(
        title="Side-by-side comparison",
        show_header=True,
        header_style="bold",
        expand=True,
        show_lines=True,
    )
    t.add_column("Metric",        style="dim",    min_width=20)
    t.add_column("Base Agent",    style="yellow", min_width=40)
    t.add_column("Adapted Agent", style="green",  min_width=40)

    t.add_row(
        "Model",
        "mistral-large-latest",
        f"mistral-large-latest [dim](loop)[/dim]\n+ {_cls_model} [dim](classifier)[/dim]",
    )
    t.add_row(
        "Parameters",
        "~123B",
        f"~123B [dim](loop)[/dim]\n+ {_cls_params} [dim](classifier)[/dim]",
    )
    t.add_row(
        "Tokens (in / out)",
        f"{base_prompt:,} / {base_completion:,}",
        (
            f"{adpt_prompt:,} / {adpt_completion:,} [dim](loop)[/dim]\n"
            f"+ {cls_prompt:,} / {cls_completion:,} [dim](classifier)[/dim]"
        ),
    )
    t.add_row(
        "Inference cost",
        f"${base_cost:.4f}",
        (
            f"${adapted_cost:.4f} total\n"
            f"[dim](${_compute_cost(adpt_prompt, adpt_completion):.4f} loop"
            f" + ${_compute_cost(0, 0, cls_prompt, cls_completion):.4f} classifier)[/dim]"
        ),
    )
    _base_tick    = "[green]✓[/green]" if base_correct    else "[red]✗[/red]"
    _adapted_tick = "[green]✓[/green]" if adapted_correct else "[red]✗[/red]"
    t.add_row(
        "Intent",
        f"{base_intent} {_base_tick}\n[dim](inferred from tool args)[/dim]",
        f"[bold]{adapted_intent}[/bold] {_adapted_tick}\n[dim]{adapted_conf:.0%} confidence[/dim]",
    )
    t.add_row(
        "Ground truth",
        f"[dim]{ground_truth}[/dim]",
        f"[dim]{ground_truth}[/dim]",
    )
    t.add_row(
        "Classifier confidence",
        "[dim]N/A[/dim]",
        f"{adapted_conf:.2f}",
    )
    t.add_row(
        "Tools called",
        "\n".join(base_result["tools_called"]) if base_result["tools_called"] else "[dim]none[/dim]",
        "\n".join(adapted_result["tools_called"]) if adapted_result["tools_called"] else "[dim]none[/dim]",
    )
    t.add_row(
        "LLM calls",
        str(base_result["llm_calls"]),
        str(adapted_result["llm_calls"]),
    )
    t.add_row(
        "Total latency",
        f"{base_result['latency_ms']:.0f} ms",
        (
            f"{adapted_result['latency_ms']:.0f} ms total\n"
            f"[dim]({adapted_result['classifier_latency_ms']:.0f} ms classify + "
            f"{adapted_result['latency_ms'] - adapted_result['classifier_latency_ms']:.0f} ms loop)[/dim]"
        ),
    )
    t.add_row(
        "Ticket created",
        base_result["ticket_id"] or "[dim]—[/dim]",
        adapted_result["ticket_id"] or "[dim]—[/dim]",
    )
    t.add_row(
        "Response preview",
        _preview(base_result["response"]),
        _preview(adapted_result["response"]),
    )

    console.print(t)

    # ── "What the SFT classifier changed" panel ────────────────────────────
    tools_saved  = len(base_result["tools_called"]) - len(adapted_result["tools_called"])
    latency_diff = base_result["latency_ms"] - adapted_result["latency_ms"]
    cost_saved   = base_cost - adapted_cost
    direction    = "faster" if latency_diff > 0 else "slower"

    console.print(Panel(
        (
            f"Tool calls saved:    [bold]{tools_saved:+d}[/bold]"
            f"  (base={len(base_result['tools_called'])}  adapted={len(adapted_result['tools_called'])})\n"
            f"Latency delta:       [bold]{latency_diff:+.0f} ms[/bold]  ({direction} with classifier)\n"
            f"Cost delta:          [bold]${cost_saved:+.4f}[/bold]  ({'cheaper' if cost_saved > 0 else 'costlier'} with classifier)\n"
            f"Base intent:         {_base_tick}  {base_intent}\n"
            f"Adapted intent:      {_adapted_tick}  {adapted_intent} [dim]({adapted_conf:.0%} confidence)[/dim]\n"
            f"Ground truth:        [bold]{ground_truth}[/bold]"
        ),
        title="[bold]What the SFT classifier changed[/bold]",
        border_style="cyan",
    ))
    console.print()

    return {
        "name":                       scenario["name"],
        "base_tool_calls":            len(base_result["tools_called"]),
        "adapted_tool_calls":         len(adapted_result["tools_called"]),
        "base_llm_calls":             base_result["llm_calls"],
        "adapted_llm_calls":          adapted_result["llm_calls"],
        "base_latency_ms":            base_result["latency_ms"],
        "adapted_latency_ms":         adapted_result["latency_ms"],
        "classifier_latency_ms":      adapted_result["classifier_latency_ms"],
        "classifier_intent":          adapted_intent,
        "classifier_confidence":      adapted_conf,
        "ground_truth":               ground_truth,
        "base_intent_correct":        base_correct,
        "adapted_intent_correct":     adapted_correct,
        "ticket_created":             adapted_result["ticket_id"] is not None,
        "base_cost_usd":              base_cost,
        "adapted_cost_usd":           adapted_cost,
        "base_prompt_tokens":         base_prompt,
        "base_completion_tokens":     base_completion,
        "adapted_prompt_tokens":      adpt_prompt,
        "adapted_completion_tokens":  adpt_completion,
    }

## 5. Test set evaluation

Both agents are run on every example in the held-out test set.
Results are collected silently then displayed as a per-class accuracy
table and an aggregate metrics panel.


In [7]:
results = []
for i, rec in enumerate(test_records, 1):
    r = _evaluate_pair(rec["text"], rec["ground_truth"])
    results.append(r)
    _b = "[green]✓[/green]" if r["base_correct"]    else "[red]✗[/red]"
    _a = "[green]✓[/green]" if r["adapted_correct"] else "[red]✗[/red]"
    console.print(
        f"  [{i:3d}/{len(test_records)}]  "
        f"gt={r['ground_truth']:<22}  "
        f"base={_b}  adapted={_a}  {r['adapted_intent']}"
    )

df = pd.DataFrame(results)

# ── Per-class accuracy table ──────────────────────────────────────────────
intent_labels = sorted(df["ground_truth"].unique())

acc_table = Table(
    title="Per-class accuracy — held-out test set",
    header_style="bold",
    show_lines=True,
    expand=True,
)
acc_table.add_column("Intent",          style="cyan",   min_width=22)
acc_table.add_column("N",               justify="right")
acc_table.add_column("Base\ncorrect",   justify="right", style="yellow")
acc_table.add_column("Base\nacc",       justify="right", style="yellow")
acc_table.add_column("Adapted\ncorrect",justify="right", style="green")
acc_table.add_column("Adapted\nacc",    justify="right", style="green")

for intent in intent_labels:
    rows = df[df["ground_truth"] == intent]
    n  = len(rows)
    bc = int(rows["base_correct"].sum())
    ac = int(rows["adapted_correct"].sum())
    acc_table.add_row(
        intent, str(n),
        str(bc), f"{bc/n:.0%}",
        str(ac), f"{ac/n:.0%}",
    )

n_total  = len(df)
bc_total = int(df["base_correct"].sum())
ac_total = int(df["adapted_correct"].sum())
acc_table.add_row(
    "[bold]TOTAL[/bold]", f"[bold]{n_total}[/bold]",
    f"[bold]{bc_total}[/bold]", f"[bold]{bc_total/n_total:.0%}[/bold]",
    f"[bold]{ac_total}[/bold]",  f"[bold]{ac_total/n_total:.0%}[/bold]",
)
console.print(acc_table)

# ── Aggregate metrics panel ───────────────────────────────────────────────
avg_tools_saved   = df["base_tool_calls"].mean()  - df["adapted_tool_calls"].mean()
avg_latency_delta = df["base_latency_ms"].mean()  - df["adapted_latency_ms"].mean()
avg_cost_saved    = df["base_cost_usd"].mean()    - df["adapted_cost_usd"].mean()
direction = "faster" if avg_latency_delta > 0 else "slower"

console.print(Panel(
    (
        f"Base intent accuracy:             [bold yellow]{bc_total/n_total:.0%}[/bold yellow]"        f"  ({bc_total}/{n_total})\n"
        f"Adapted intent accuracy:          [bold green]{ac_total/n_total:.0%}[/bold green]"        f"  ({ac_total}/{n_total})\n"
        f"Avg tool calls saved per query:   [bold]{avg_tools_saved:+.1f}[/bold]\n"
        f"Avg latency delta:                [bold]{avg_latency_delta:+.0f} ms[/bold]"        f"  ({direction} with classifier)\n"
        f"Avg cost delta per query:         [bold]${avg_cost_saved:+.4f}[/bold]"        f"  ({'cheaper' if avg_cost_saved > 0 else 'costlier'} with classifier)"
    ),
    title="[bold]Key numbers — full test set[/bold]",
    border_style="cyan",
))


[  1/64]  gt=payments_incident       base=✓  adapted=✓  payments_incident

[  2/64]  gt=security_incident       base=✓  adapted=✓  security_incident

[  3/64]  gt=general_question        base=✓  adapted=✓  general_question

[  4/64]  gt=general_question        base=✓  adapted=✓  general_question

[  5/64]  gt=security_incident       base=✓  adapted=✗  hardware_issue

...

[ 62/64]  gt=software_issue          base=✓  adapted=✓  software_issue

[ 63/64]  gt=payments_incident       base=✓  adapted=✓  payments_incident

[ 64/64]  gt=hardware_issue          base=✗  adapted=✓  hardware_issue

                                         Per-class accuracy — held-out test set                                         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓
┃                                             ┃        ┃            Base ┃      Base ┃        Adapted ┃        Adapted ┃
┃ Intent                                      ┃      N ┃         correct ┃       acc ┃        correct ┃            acc ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩
│ access_request                              │      7 │               7 │      100% │              7 │           100% │
├─────────────────────────────────────────────┼────────┼─────────────────┼───────────┼────────────────┼────────────────┤
│ expense_request                             │      8 │               5 │       62% │              8 │           100% │
├─────────────────────────────────────────────┼────────┼─────────────────┼───────────┼────────────────┼────────────────┤
│ general_question                            │      8 │               5 │       62% │              6 │            75% │
├─────────────────────────────────────────────┼────────┼─────────────────┼───────────┼────────────────┼────────────────┤
│ hardware_issue                              │      9 │               6 │       67% │              8 │            89% │
├─────────────────────────────────────────────┼────────┼─────────────────┼───────────┼────────────────┼────────────────┤
│ onboarding                                  │      8 │               1 │       12% │              7 │            88% │
├─────────────────────────────────────────────┼────────┼─────────────────┼───────────┼────────────────┼────────────────┤
│ payments_incident                           │      8 │               8 │      100% │              8 │           100% │
├─────────────────────────────────────────────┼────────┼─────────────────┼───────────┼────────────────┼────────────────┤
│ security_incident                           │      8 │               8 │      100% │              7 │            88% │
├─────────────────────────────────────────────┼────────┼─────────────────┼───────────┼────────────────┼────────────────┤
│ software_issue                              │      8 │               6 │       75% │              7 │            88% │
├─────────────────────────────────────────────┼────────┼─────────────────┼───────────┼────────────────┼────────────────┤
│ TOTAL                                       │     64 │              46 │       72% │             58 │            91% │
└─────────────────────────────────────────────┴────────┴─────────────────┴───────────┴────────────────┴────────────────┘

╭──────────────────────────────────────────── Key numbers — full test set ─────────────────────────────────────────────╮
│ Base intent accuracy:             72%  (46/64)                                                                       │
│ Adapted intent accuracy:          91%  (58/64)                                                                       │
│ Avg tool calls saved per query:   +2.6                                                                               │
│ Avg latency delta:                +3428 ms  (faster with classifier)                                                 │
│ Avg cost delta per query:         $+0.0119  (cheaper with classifier)                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 6. Interactive cell

Type any custom IT request and run it through both agents live.

In [ ]:
query_box = widgets.Textarea(
    value="",
    placeholder="Type a custom IT request here and press Run…",
    layout=widgets.Layout(width="100%", height="90px"),
)

run_btn = widgets.Button(
    description="Run comparison",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(width="180px"),
)

output_area = widgets.Output()


def on_run(_button):
    with output_area:
        clear_output(wait=True)
        text = query_box.value.strip()
        if not text:
            print("Please enter a request first.")
            return
        run_comparison({"name": "Custom query", "text": text})


run_btn.on_click(on_run)
display(query_box, run_btn, output_area)

## 7. What we just saw — and what it means for Forge

### The three-phase adapted agent loop

Every adapted-agent query went through three phases:

1. **Classification** — the fine-tuned `Mistral-7B` model read the ticket text and returned
   an intent label in a single forward pass (~50–150 ms). No LLM sampling,
   no chain-of-thought — the model outputs exactly one of 8 intent labels at `temperature=0`.

2. **Tool restriction** — the router swapped the 3-tool schema for a 2-tool schema, removing
   `search_knowledge_base` from the context window. The system prompt was also primed with the
   intent, so the model knew immediately which escalation path to check.

3. **Agentic loop** — `mistral-large` saw a shorter tool schema and a pre-filled intent. It
   skipped exploratory KB searches and went straight to `get_escalation_policy` or
   `create_ticket`. Fewer tokens in, fewer tool-call round-trips out.

The result: explicitly logged routing decisions that are auditable, fewer prompt tokens per
turn, and lower end-to-end latency even after the classifier overhead is accounted for.

---

### SFT fine-tuning vs. Forge — the same principle at two scales

| | SFT classifier (what we ran) | Forge continued pre-training |
|---|---|---|
| **Training data** | Labelled examples `{"messages": [{role: system}, {role: user}, {role: assistant → label}]}` | Unlabelled domain corpus (tickets, runbooks, wikis) |
| **What changes in the model** | Model learns to output the correct intent label | Full model weights across all layers |
| **Output at inference** | Discrete intent label via `client.chat.complete()` at `temperature=0` | Richer domain representations at every token |
| **Inference call** | `client.chat.complete()` with the fine-tuned model ID | `client.chat.complete()` via your Forge endpoint URL |
| **Value** | Accurate, fast routing on your label taxonomy | Institutional fluency — Nexus, Prism, Helix, SEV1, prod-payments are first-class concepts |

The SFT fine-tune we ran here demonstrates the supervised, label-aware variant of the fine-tune-on-proprietary-data
principle. Forge adds data-residency guarantees, dedicated infrastructure, and the ability to
pre-train on unlabelled corpora so the generative model itself develops institutional fluency —
not just routing accuracy. Switching from La Plateforme to Forge is a single parameter change:

```python
# La Plateforme (this demo)
client = Mistral(api_key=MISTRAL_API_KEY)

# Forge deployment
client = Mistral(api_key=FORGE_API_KEY, server_url="https://mistral.your-forge.internal/v1")
```

---

### Next steps

**To go deeper with what we built:**
- Scale the training set to 100–200 examples per class for production-grade accuracy
- Add an `unknown` class to catch out-of-distribution queries gracefully
- Log `classifier_intent` and `classifier_confidence` per request; feed misclassified
  examples back into the training set automatically

**Mistral documentation:**
- Fine-tuning API reference: `https://docs.mistral.ai/api/#tag/fine-tuning`
- Mistral Forge (contact sales): `https://mistral.ai/products/la-plateforme`